In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import time

project_path = "/content/drive/MyDrive/Enhanced_IDS_Project"

print("Project path:", project_path)

Project path: /content/drive/MyDrive/Enhanced_IDS_Project


In [ ]:
print("Files in processed_data:")

for file in os.listdir(project_path + "/processed_data"):
    print(file)

Files in processed_data:
X_train_normalized.npy
X_test_normalized.npy
y_train.npy
y_test.npy
encoder.pkl
train_mean.npy
train_range.npy


In [ ]:
X_train = np.load(
    project_path + "/processed_data/X_train_normalized.npy"
)

X_test = np.load(
    project_path + "/processed_data/X_test_normalized.npy"
)

y_train = np.load(
    project_path + "/processed_data/y_train.npy"
)

y_test = np.load(
    project_path + "/processed_data/y_test.npy"
)

print("Data loaded successfully!")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Data loaded successfully!
X_train: (125973, 122)
X_test : (22543, 122)
y_train: (125973,)
y_test : (22543,)


In [ ]:
# MVO parameters

num_features = X_train.shape[1]

num_universes = 10
max_iterations = 20

alpha = 0.9

print("Number of features:", num_features)
print("Number of universes:", num_universes)
print("Maximum iterations:", max_iterations)
print("Alpha:", alpha)

Number of features: 122
Number of universes: 10
Maximum iterations: 20
Alpha: 0.9


In [ ]:
# Create initial MVO population

np.random.seed(42)

universes = np.random.randint(
    0,
    2,
    size=(num_universes, num_features)
)

print("Universes shape:", universes.shape)
print("\nFirst universe:")
print(universes[0])

print("\nNumber of selected features in first universe:",
      np.sum(universes[0]))

Universes shape: (10, 122)

First universe:
[0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1
 0 0 0 0 0 1 1 1 1 1 0 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 0 1
 0 1 1 1 0 1 0 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 0 1
 0 1 1 0 1 0 1 1 0 1 0]

Number of selected features in first universe: 71


In [ ]:
# MVO Fitness Function

def calculate_fitness(feature_mask):

    # Make sure at least one feature is selected
    if np.sum(feature_mask) == 0:
        return 1.0

    # Select only the features marked as 1
    selected_features = feature_mask.astype(bool)

    X_train_selected = X_train[:, selected_features]
    X_test_selected = X_test[:, selected_features]

    # Lightweight classifier for MVO evaluation
    model = LogisticRegression(
        max_iter=100,
        solver="liblinear",
        random_state=42
    )

    # Train
    model.fit(X_train_selected, y_train)

    # Predict
    predictions = model.predict(X_test_selected)

    # Classification error
    error = 1 - accuracy_score(y_test, predictions)

    # Feature selection ratio
    feature_ratio = np.sum(feature_mask) / num_features

    # Final fitness
    fitness = (
        alpha * error +
        (1 - alpha) * feature_ratio
    )

    return fitness

print("Fitness function created successfully!")

Fitness function created successfully!


In [ ]:
# Test fitness function on the first universe

first_universe = universes[0]

fitness_value = calculate_fitness(first_universe)

print("Selected features:", np.sum(first_universe))
print("Total features:", num_features)
print("Fitness value:", fitness_value)

Selected features: 71
Total features: 122
Fitness value: 0.26915355935432683


In [ ]:
# Calculate fitness for all initial universes

fitness_values = np.zeros(num_universes)

for i in range(num_universes):
    fitness_values[i] = calculate_fitness(universes[i])
    print(
        "Universe", i + 1,
        "| Selected features:", np.sum(universes[i]),
        "| Fitness:", fitness_values[i]
    )

Universe 1 | Selected features: 71 | Fitness: 0.26915355935432683
Universe 2 | Selected features: 52 | Fitness: 0.24399810053355223
Universe 3 | Selected features: 59 | Fitness: 0.2502148898680336
Universe 4 | Selected features: 70 | Fitness: 0.2944839116209968
Universe 5 | Selected features: 66 | Fitness: 0.27375856559740475
Universe 6 | Selected features: 65 | Fitness: 0.2661518642332359
Universe 7 | Selected features: 53 | Fitness: 0.2498880827387804
Universe 8 | Selected features: 54 | Fitness: 0.27765625329515975
Universe 9 | Selected features: 59 | Fitness: 0.2585988671558835
Universe 10 | Selected features: 61 | Fitness: 0.2497782016590516


In [ ]:
# Find the best universe

best_index = np.argmin(fitness_values)

best_universe = universes[best_index].copy()
best_fitness = fitness_values[best_index]

print("Best universe:", best_index + 1)
print("Best fitness:", best_fitness)
print("Selected features:", np.sum(best_universe))

Best universe: 2
Best fitness: 0.24399810053355223
Selected features: 52


In [ ]:
# Calculate WEP and TDR for the first MVO iteration

iteration = 1

WEP_min = 0.2
WEP_max = 1.0

WEP = WEP_min + (
    (WEP_max - WEP_min) * iteration / max_iterations
)

TDR = 1 - (
    iteration / max_iterations
) ** (1 / 6)

print("Iteration:", iteration)
print("WEP:", WEP)
print("TDR:", TDR)

Iteration: 1
WEP: 0.24000000000000002
TDR: 0.39303776899708276


In [ ]:
# MVO universe update

new_universes = universes.copy()

for i in range(num_universes):

    for j in range(num_features):

        # White-hole / black-hole exploration
        if np.random.rand() < 0.5:

            # Select a random universe
            donor = np.random.randint(0, num_universes)

            # Copy the donor's feature
            new_universes[i, j] = universes[donor, j]

        # Wormhole exploitation
        if np.random.rand() < WEP:

            if np.random.rand() < 0.5:
                new_universes[i, j] = best_universe[j]

            else:
                # Occasionally flip the feature
                if np.random.rand() < TDR:
                    new_universes[i, j] = 1 - new_universes[i, j]

# Make sure no universe has zero selected features
for i in range(num_universes):

    if np.sum(new_universes[i]) == 0:
        random_feature = np.random.randint(0, num_features)
        new_universes[i, random_feature] = 1

print("Universe update completed!")
print("New population shape:", new_universes.shape)

Universe update completed!
New population shape: (10, 122)


In [ ]:
# Calculate fitness of the updated universes

new_fitness_values = np.zeros(num_universes)

for i in range(num_universes):
    new_fitness_values[i] = calculate_fitness(new_universes[i])

    print(
        "Universe", i + 1,
        "| Selected features:", np.sum(new_universes[i]),
        "| Fitness:", new_fitness_values[i]
    )

Universe 1 | Selected features: 67 | Fitness: 0.27453831402718154
Universe 2 | Selected features: 64 | Fitness: 0.2765108284858882
Universe 3 | Selected features: 54 | Fitness: 0.25521913312481864
Universe 4 | Selected features: 63 | Fitness: 0.27752764661779344
Universe 5 | Selected features: 53 | Fitness: 0.2756787938242615
Universe 6 | Selected features: 59 | Fitness: 0.2526502356516472
Universe 7 | Selected features: 61 | Fitness: 0.2573237812181165
Universe 8 | Selected features: 53 | Fitness: 0.2860589561806471
Universe 9 | Selected features: 75 | Fitness: 0.2772230920434026
Universe 10 | Selected features: 62 | Fitness: 0.24836214651343919


In [ ]:
# Keep the better population

for i in range(num_universes):

    if new_fitness_values[i] < fitness_values[i]:
        universes[i] = new_universes[i].copy()
        fitness_values[i] = new_fitness_values[i]

# Find the best universe after selection
current_best_index = np.argmin(fitness_values)
current_best_fitness = fitness_values[current_best_index]

# Update global best only if it improved
if current_best_fitness < best_fitness:
    best_fitness = current_best_fitness
    best_universe = universes[current_best_index].copy()

print("Current best fitness:", current_best_fitness)
print("Global best fitness:", best_fitness)
print("Selected features:", np.sum(best_universe))

Current best fitness: 0.24399810053355223
Global best fitness: 0.24399810053355223
Selected features: 52


In [ ]:
# ============================================================
# MVO FEATURE SELECTION - 20 ITERATIONS
# ============================================================

# Reset to the original population
np.random.seed(42)

universes = np.random.randint(
    0,
    2,
    size=(num_universes, num_features)
)

# Make sure every universe has at least one selected feature
for i in range(num_universes):
    if np.sum(universes[i]) == 0:
        random_feature = np.random.randint(0, num_features)
        universes[i, random_feature] = 1

# Calculate initial fitness
fitness_values = np.zeros(num_universes)

for i in range(num_universes):
    fitness_values[i] = calculate_fitness(universes[i])

# Initial best solution
best_index = np.argmin(fitness_values)

best_universe = universes[best_index].copy()
best_fitness = fitness_values[best_index]

print("Initial best fitness:", best_fitness)
print("Initial selected features:", np.sum(best_universe))
print()

# ------------------------------------------------------------
# MVO ITERATIONS
# ------------------------------------------------------------

for iteration in range(1, max_iterations + 1):

    # Calculate WEP
    WEP = WEP_min + (
        (WEP_max - WEP_min)
        * iteration
        / max_iterations
    )

    # Calculate TDR
    TDR = 1 - (
        iteration / max_iterations
    ) ** (1 / 6)

    # Create new population
    new_universes = universes.copy()

    # --------------------------------------------------------
    # Update every universe
    # --------------------------------------------------------

    for i in range(num_universes):

        for j in range(num_features):

            # White-hole / black-hole operation
            if np.random.rand() < 0.5:

                donor = np.random.randint(
                    0,
                    num_universes
                )

                new_universes[i, j] = universes[donor, j]

            # Wormhole operation
            if np.random.rand() < WEP:

                if np.random.rand() < 0.5:

                    # Move towards best universe
                    new_universes[i, j] = best_universe[j]

                else:

                    # Feature mutation
                    if np.random.rand() < TDR:
                        new_universes[i, j] = (
                            1 - new_universes[i, j]
                        )

    # Make sure no universe has zero features
    for i in range(num_universes):

        if np.sum(new_universes[i]) == 0:

            random_feature = np.random.randint(
                0,
                num_features
            )

            new_universes[i, random_feature] = 1

    # --------------------------------------------------------
    # Evaluate new population
    # --------------------------------------------------------

    new_fitness_values = np.zeros(num_universes)

    for i in range(num_universes):

        new_fitness_values[i] = calculate_fitness(
            new_universes[i]
        )

    # --------------------------------------------------------
    # Keep better solutions
    # --------------------------------------------------------

    for i in range(num_universes):

        if new_fitness_values[i] < fitness_values[i]:

            universes[i] = new_universes[i].copy()

            fitness_values[i] = (
                new_fitness_values[i]
            )

    # --------------------------------------------------------
    # Update global best
    # --------------------------------------------------------

    current_best_index = np.argmin(
        fitness_values
    )

    current_best_fitness = (
        fitness_values[current_best_index]
    )

    if current_best_fitness < best_fitness:

        best_fitness = current_best_fitness

        best_universe = (
            universes[current_best_index].copy()
        )

    # --------------------------------------------------------
    # Print progress
    # --------------------------------------------------------

    print(
        "Iteration:", iteration,
        "| Best fitness:", best_fitness,
        "| Selected features:",
        np.sum(best_universe)
    )

print("\nMVO completed!")
print("Final best fitness:", best_fitness)
print("Final selected features:", np.sum(best_universe))

Initial best fitness: 0.24399810053355223
Initial selected features: 52

Iteration: 1 | Best fitness: 0.24399810053355223 | Selected features: 52
Iteration: 2 | Best fitness: 0.23571764125827294 | Selected features: 55
Iteration: 3 | Best fitness: 0.23236405034313293 | Selected features: 55
Iteration: 4 | Best fitness: 0.22034948873664392 | Selected features: 57
Iteration: 5 | Best fitness: 0.22034948873664392 | Selected features: 57
Iteration: 6 | Best fitness: 0.18970681168157316 | Selected features: 56
Iteration: 7 | Best fitness: 0.18970681168157316 | Selected features: 56
Iteration: 8 | Best fitness: 0.1806278783788795 | Selected features: 59
Iteration: 9 | Best fitness: 0.1789722810250428 | Selected features: 60
Iteration: 10 | Best fitness: 0.16676940899105022 | Selected features: 55
Iteration: 11 | Best fitness: 0.16676940899105022 | Selected features: 55
Iteration: 12 | Best fitness: 0.16676940899105022 | Selected features: 55


KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import train_test_split

# Use a smaller stratified dataset for MVO fitness evaluation

mvo_sample_size = 10000

X_mvo, _, y_mvo, _ = train_test_split(
    X_train,
    y_train,
    train_size=mvo_sample_size,
    stratify=y_train,
    random_state=42
)

# Split MVO data into training and validation
X_mvo_train, X_mvo_val, y_mvo_train, y_mvo_val = train_test_split(
    X_mvo,
    y_mvo,
    test_size=0.2,
    stratify=y_mvo,
    random_state=42
)

print("MVO training shape:", X_mvo_train.shape)
print("MVO validation shape:", X_mvo_val.shape)
print("MVO training labels:", y_mvo_train.shape)
print("MVO validation labels:", y_mvo_val.shape)

MVO training shape: (8000, 122)
MVO validation shape: (2000, 122)
MVO training labels: (8000,)
MVO validation labels: (2000,)


In [ ]:
# ============================================================
# UPDATED MVO FITNESS FUNCTION
# ============================================================

def calculate_fitness(feature_mask):

    # Make sure at least one feature is selected
    if np.sum(feature_mask) == 0:
        return 1.0

    # Select features
    selected_features = feature_mask.astype(bool)

    X_train_selected = X_mvo_train[:, selected_features]
    X_val_selected = X_mvo_val[:, selected_features]

    # Lightweight classifier
    model = LogisticRegression(
        max_iter=100,
        solver="liblinear",
        random_state=42
    )

    # Train
    model.fit(
        X_train_selected,
        y_mvo_train
    )

    # Validation prediction
    predictions = model.predict(
        X_val_selected
    )

    # Classification error
    error = 1 - accuracy_score(
        y_mvo_val,
        predictions
    )

    # Feature selection ratio
    feature_ratio = (
        np.sum(feature_mask) / num_features
    )

    # Final fitness
    fitness = (
        alpha * error +
        (1 - alpha) * feature_ratio
    )

    return fitness


print("Updated fitness function created successfully!")

Updated fitness function created successfully!


In [ ]:
# ============================================================
# OPTIMIZED MVO FEATURE SELECTION
# ============================================================

np.random.seed(42)

# Create initial universes
universes = np.random.randint(
    0,
    2,
    size=(num_universes, num_features)
)

# Make sure every universe selects at least one feature
for i in range(num_universes):
    if np.sum(universes[i]) == 0:
        random_feature = np.random.randint(0, num_features)
        universes[i, random_feature] = 1

# Calculate initial fitness
fitness_values = np.zeros(num_universes)

for i in range(num_universes):
    fitness_values[i] = calculate_fitness(universes[i])

# Find initial best universe
best_index = np.argmin(fitness_values)

best_universe = universes[best_index].copy()
best_fitness = fitness_values[best_index]

print("Initial best fitness:", best_fitness)
print("Initial selected features:", np.sum(best_universe))
print()

# ============================================================
# MVO ITERATIONS
# ============================================================

for iteration in range(1, max_iterations + 1):

    # WEP
    WEP = WEP_min + (
        (WEP_max - WEP_min)
        * iteration
        / max_iterations
    )

    # TDR
    TDR = 1 - (
        iteration / max_iterations
    ) ** (1 / 6)

    # Create new population
    new_universes = universes.copy()

    # Update universes
    for i in range(num_universes):

        for j in range(num_features):

            # White-hole / black-hole operation
            if np.random.rand() < 0.5:

                donor = np.random.randint(
                    0,
                    num_universes
                )

                new_universes[i, j] = universes[donor, j]

            # Wormhole operation
            if np.random.rand() < WEP:

                if np.random.rand() < 0.5:

                    # Move toward best universe
                    new_universes[i, j] = best_universe[j]

                else:

                    # Mutation
                    if np.random.rand() < TDR:
                        new_universes[i, j] = (
                            1 - new_universes[i, j]
                        )

    # Prevent empty feature subsets
    for i in range(num_universes):

        if np.sum(new_universes[i]) == 0:

            random_feature = np.random.randint(
                0,
                num_features
            )

            new_universes[i, random_feature] = 1

    # Evaluate new population
    new_fitness_values = np.zeros(num_universes)

    for i in range(num_universes):

        new_fitness_values[i] = calculate_fitness(
            new_universes[i]
        )

    # Keep better universes
    for i in range(num_universes):

        if new_fitness_values[i] < fitness_values[i]:

            universes[i] = new_universes[i].copy()

            fitness_values[i] = (
                new_fitness_values[i]
            )

    # Update global best
    current_best_index = np.argmin(
        fitness_values
    )

    current_best_fitness = (
        fitness_values[current_best_index]
    )

    if current_best_fitness < best_fitness:

        best_fitness = current_best_fitness

        best_universe = (
            universes[current_best_index].copy()
        )

    print(
        "Iteration:", iteration,
        "| Best fitness:", round(best_fitness, 6),
        "| Selected features:",
        np.sum(best_universe)
    )

print("\nMVO completed!")
print("Final best fitness:", best_fitness)
print("Final selected features:", np.sum(best_universe))

Initial best fitness: 0.07626065573770494
Initial selected features: 59

Iteration: 1 | Best fitness: 0.076261 | Selected features: 59
Iteration: 2 | Best fitness: 0.076261 | Selected features: 59
Iteration: 3 | Best fitness: 0.076261 | Selected features: 59
Iteration: 4 | Best fitness: 0.073802 | Selected features: 56
Iteration: 5 | Best fitness: 0.073802 | Selected features: 56
Iteration: 6 | Best fitness: 0.072773 | Selected features: 52
Iteration: 7 | Best fitness: 0.070764 | Selected features: 49
Iteration: 8 | Best fitness: 0.069816 | Selected features: 44
Iteration: 9 | Best fitness: 0.069494 | Selected features: 48
Iteration: 10 | Best fitness: 0.064367 | Selected features: 39
Iteration: 11 | Best fitness: 0.064367 | Selected features: 39
Iteration: 12 | Best fitness: 0.063837 | Selected features: 40
Iteration: 13 | Best fitness: 0.060478 | Selected features: 37
Iteration: 14 | Best fitness: 0.060478 | Selected features: 37
Iteration: 15 | Best fitness: 0.060478 | Selected feat

In [ ]:
# ============================================================
# FINAL MVO RESULT
# ============================================================

selected_feature_indices = np.where(
    best_universe == 1
)[0]

print("Total original features:", num_features)
print("Selected features:", len(selected_feature_indices))
print("Final fitness:", best_fitness)

print("\nSelected feature indices:")
print(selected_feature_indices)

Total original features: 122
Selected features: 33
Final fitness: 0.05539918032786882

Selected feature indices:
[  7   8   9  13  17  20  21  22  23  24  28  29  31  36  37  40  48  51
  52  54  55  61  65  68  75  90  94  95 103 104 113 118 119]


In [ ]:
# ============================================================
# SAVE MVO RESULTS
# ============================================================

mvo_path = project_path + "/mvo_results"

os.makedirs(mvo_path, exist_ok=True)

# Save selected feature indices
np.save(
    mvo_path + "/selected_feature_indices.npy",
    selected_feature_indices
)

# Save best feature mask
np.save(
    mvo_path + "/best_feature_mask.npy",
    best_universe
)

# Save final fitness
np.save(
    mvo_path + "/best_fitness.npy",
    np.array([best_fitness])
)

print("MVO results saved successfully!")

print("\nSaved files:")
print("1. selected_feature_indices.npy")
print("2. best_feature_mask.npy")
print("3. best_fitness.npy")

MVO results saved successfully!

Saved files:
1. selected_feature_indices.npy
2. best_feature_mask.npy
3. best_fitness.npy
